# CodeAct and RLM

This notebook covers Chapter 7 sections 7.1.8 (`dspy.CodeAct`) and 7.1.9 (`dspy.RLM`, Recursive Language Models).

- **CodeAct** sits at the intersection of `ReAct` and `ProgramOfThought`: it generates Python code that calls your tools directly.
- **RLM** puts the model in a sandboxed Python REPL with your context stored as variables. The model writes code to explore the data and can call a cheaper sub-LM for semantic chunks.

**Environment variables**: `OPENAI_API_KEY` (loaded from `.env`).

## Deno is required

Both modules execute code in a Deno + Pyodide sandbox. Install Deno before running this notebook (in a terminal, not as a notebook cell):

```
curl -fsSL https://deno.land/install.sh | sh
```

RLM is marked as `@experimental` in DSPy and requires a recent version (post January 2026).


In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))


## 7.1.8 CodeAct

Tools must be plain Python functions (not callable objects) because they get injected into the sandboxed interpreter so the model can call them in its generated code.


In [ ]:
import dspy

def get_population(city: str) -> int:
    """Get the current population of a city."""
    populations = {
        "Tokyo": 13_960_000, "Delhi": 32_940_000,
        "Shanghai": 28_520_000, "São Paulo": 22_430_000,
        "Mumbai": 21_670_000
    }
    return populations.get(city, 0)

def get_temperature(city: str) -> float:
    """Get the current temperature in Celsius for a city."""
    temps = {"Tokyo": 18.5, "Delhi": 35.2, "Shanghai": 22.1,
             "São Paulo": 25.8, "Mumbai": 31.4}
    return temps.get(city, 0.0)

agent = dspy.CodeAct(
    "question: str -> answer: str",
    tools=[get_population, get_temperature],
    max_iters=5
)

result = agent(
    question="Which of the top 5 most populated cities is currently the coldest?"
)


## 7.1.9 RLM (Recursive Language Models)

The main LM handles strategy. A cheaper sub-LM handles extraction and is called via the built-in `llm_query` and `llm_query_batched` tools. The model submits its final answer with `SUBMIT(output)`.

The snippet below assumes a `contract.txt` file is present locally. Drop in any long document of your choice (50,000+ tokens is the kind of context size where RLM shines).


In [ ]:
import dspy

# Use a cheap model for the recursive sub-queries
sub_lm = dspy.LM("openai/gpt-4o-mini")

analyzer = dspy.RLM(
    "document: str, question: str -> answer: str",
    max_iterations=20,
    max_llm_calls=50,
    sub_lm=sub_lm
)

long_contract = open("contract.txt").read()  # 50,000+ tokens
result = analyzer(
    document=long_contract,
    question="What are the termination conditions and associated penalties?"
)
